In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Seller metrics validation").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
seller_metrics = spark.read.parquet("../../data/gold/seller_metrics")

In [4]:
seller_metrics.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_products_sold: long (nullable = true)
 |-- unique_products: long (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- average_product_price: double (nullable = true)
 |-- total_freight: double (nullable = true)
 |-- first_sale_date: date (nullable = true)
 |-- last_sale_date: date (nullable = true)
 |-- seller_rank: integer (nullable = true)



In [5]:
seller_metrics.show(10,truncate = False)

+--------------------------------+----------------+------------+------------+-------------------+---------------+-----------+---------------------+-------------+---------------+--------------+-----------+
|seller_id                       |seller_city     |seller_state|total_orders|total_products_sold|unique_products|total_sales|average_product_price|total_freight|first_sale_date|last_sale_date|seller_rank|
+--------------------------------+----------------+------------+------------+-------------------+---------------+-----------+---------------------+-------------+---------------+--------------+-----------+
|4869f7a5dfa277a7dca6462dcf3b52b2|guariba         |SP          |1132        |1156               |95             |229472.63  |198.51               |20168.07     |2017-03-07     |2018-08-24    |1          |
|53243585a1d6dc2643021fd1853d8905|lauro de freitas|BA          |358         |410                |23             |222776.05  |543.36               |13080.63     |2017-08-28     |201

In [6]:
seller_metrics.count()

3095

In [7]:
seller_metrics.select("seller_id").distinct().count()

3095

In [19]:
from pyspark.sql.functions import col,sum

In [10]:
seller_metrics.filter(col("seller_id").isNull()).count()

0

In [11]:
seller_metrics.filter(col("total_sales").isNull()).count()

0

In [12]:
seller_metrics.filter(col("total_products_Sold") <=0).count()

0

In [13]:
seller_metrics.orderBy(col("seller_rank")).show(10)

+--------------------+----------------+------------+------------+-------------------+---------------+-----------+---------------------+-------------+---------------+--------------+-----------+
|           seller_id|     seller_city|seller_state|total_orders|total_products_sold|unique_products|total_sales|average_product_price|total_freight|first_sale_date|last_sale_date|seller_rank|
+--------------------+----------------+------------+------------+-------------------+---------------+-----------+---------------------+-------------+---------------+--------------+-----------+
|4869f7a5dfa277a7d...|         guariba|          SP|        1132|               1156|             95|  229472.63|               198.51|     20168.07|     2017-03-07|    2018-08-24|          1|
|53243585a1d6dc264...|lauro de freitas|          BA|         358|                410|             23|  222776.05|               543.36|     13080.63|     2017-08-28|    2018-08-21|          2|
|4a3ca9315b744ce9f...|        ibiti

In [ ]:
#source vs Gold reconcilation,, compared seller metrics total against silver order_items
seller_metrics.selectExpr(
    "sum(total_sales) as gold_sales"
).show()

+--------------------+
|          gold_sales|
+--------------------+
|1.3591643700000146E7|
+--------------------+



In [18]:
order_items = spark.read.parquet("../../data/silver/order_items")

In [21]:
order_items.selectExpr("sum(price) as silver_sales").show()

+-------------------+
|       silver_sales|
+-------------------+
|1.359164370001419E7|
+-------------------+

